# L23 OCR ingestion

Reads on-frame text (captions, signs, chyrons) from L23 keyframes with PaddleOCR.
Keyframes stay on Colab SSD. Resumable per-video JSON goes to Drive; the L23 `ocr.sqlite`
release is written to `AIC_ARTIFACTS/releases/l23-v1`, joining the existing `frames.csv`
published by `ingest_beit3.ipynb`. A GPU runtime speeds this up but is not required.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!pip -q install paddleocr paddlepaddle
!mkdir -p /content/work
!wget -q https://aic-data.ledo.io.vn/Keyframes_L23.zip -O /content/work/Keyframes_L23.zip
!unzip -q /content/work/Keyframes_L23.zip -d /content/work/keyframes

In [ ]:
import csv
import json
import shutil
import sqlite3
from pathlib import Path

from paddleocr import PaddleOCR

work = Path("/content/work")
release_name = "l23-v1"
output = Path("/content/drive/MyDrive/AIC_ARTIFACTS")
texts_dir = output / "workers" / "ocr"
release_dir = output / "releases" / release_name
texts_dir.mkdir(parents=True, exist_ok=True)
release_dir.mkdir(parents=True, exist_ok=True)

engine = PaddleOCR(use_angle_cls=True, lang="vi", show_log=False)

frame_dirs = []
for path in work.rglob("L23_V*"):
    if path.is_dir() and any(path.glob("*.jpg")):
        frame_dirs.append(path)
frame_dirs.sort(key=lambda path: path.name)

In [ ]:
def ocr_frame(image_path, min_confidence=0.5):
    pages = engine.ocr(str(image_path), cls=True)
    lines = []
    for page in pages or []:
        for _, (text, confidence) in page or []:
            if confidence >= min_confidence:
                lines.append(text)
    return " ".join(lines)


for frame_dir in frame_dirs:
    video_id = frame_dir.name
    output_path = texts_dir / f"{video_id}.json"
    if output_path.exists():
        print(f"skip {video_id}")
        continue
    frames = sorted(frame_dir.glob("*.jpg"), key=lambda path: int(path.stem))
    results = []
    for frame in frames:
        text = ocr_frame(frame)
        if text:
            results.append({"keyframe_n": int(frame.stem), "text": text})
    temporary_path = output_path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(results, ensure_ascii=False), encoding="utf-8")
    temporary_path.replace(output_path)
    print(f"saved {video_id} {len(results)} of {len(frames)} frames with text")

In [ ]:
with (release_dir / "frames.csv").open(newline="", encoding="utf-8") as file:
    frames_by_uid = {row["frame_uid"]: row for row in csv.DictReader(file)}
expected_videos = {row["video_id"] for row in frames_by_uid.values()}

text_paths = sorted(texts_dir.glob("L23_V*.json"))
text_videos = {path.stem for path in text_paths}
if not expected_videos.issubset(text_videos):
    raise ValueError(f"have {len(text_videos)} OCR outputs for {len(expected_videos)} L23 videos")

database_path = release_dir / "ocr.sqlite"
temporary_path = release_dir / "ocr.sqlite.tmp"
temporary_path.unlink(missing_ok=True)
frames_with_text = 0
with sqlite3.connect(temporary_path) as connection:
    connection.execute(
        "CREATE VIRTUAL TABLE ocr USING fts5(video_id UNINDEXED, timestamp_sec UNINDEXED, frame_path UNINDEXED, text)"
    )
    for text_path in text_paths:
        video_id = text_path.stem
        if video_id not in expected_videos:
            continue
        for item in json.loads(text_path.read_text(encoding="utf-8")):
            frame = frames_by_uid.get(f"{video_id}__kf_{item['keyframe_n']}")
            if frame is None:
                continue
            connection.execute(
                "INSERT INTO ocr(video_id, timestamp_sec, frame_path, text) VALUES (?, ?, ?, ?)",
                (video_id, float(frame["timestamp_sec"]), frame["frame_path"], item["text"]),
            )
            frames_with_text += 1
temporary_path.replace(database_path)
print(f"published {release_name}/ocr.sqlite with {frames_with_text} frames of text")
shutil.rmtree(work)

In [ ]:
query = "biển số"
terms = [f'"{term}"' for term in query.split()]
with sqlite3.connect(database_path) as connection:
    rows = connection.execute(
        "SELECT video_id, timestamp_sec, frame_path, text, bm25(ocr) FROM ocr WHERE ocr MATCH ? ORDER BY bm25(ocr) LIMIT 5",
        (" OR ".join(terms),),
    ).fetchall()
for video_id, timestamp_sec, frame_path, text, score in rows:
    print(video_id, timestamp_sec, text, score)